# Session 4 — From chat to API

We run this together. By the end you'll have made programmatic model calls, seen temperature with your own eyes, produced structured output into a table, and batched a real classification. **None of this requires writing code today** — run the cells, read what they do, change small things when invited.

In [ ]:
import sys, pathlib, time
sys.path.append(str(pathlib.Path("..").resolve()))   # notebooks/ -> repo root
import course_utils

try:
    client = course_utils.get_client(verbose=True)
except RuntimeError:
    # First time here without check_setup? We can store the key now too:
    import getpass, os
    key = getpass.getpass("Paste your API key (input stays hidden): ")
    course_utils._persist("GEMINI_API_KEY", key)
    os.environ["GEMINI_API_KEY"] = key
    client = course_utils.get_client(verbose=True)

MODEL = course_utils.MODEL
import os

# Polite-API pattern: the free tier meters ~10 requests/minute, so loops in this
# notebook pace themselves, and a rate-limit error ("429") waits and retries once
# instead of crashing. Steal this wrapper for your own projects.
PACE = 6.5 if os.environ.get("GEMINI_MODE") == "dev" else 1.0  # seconds between calls

def ask(contents, **config):
    for attempt in (1, 2):
        try:
            return client.models.generate_content(model=MODEL, contents=contents,
                                                  config=config or None)
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                if attempt == 1:
                    print("   (rate limit reached -- waiting 30s and retrying...)")
                    time.sleep(30)
                else:
                    raise RuntimeError("Still rate-limited. Wait a minute, then rerun this cell. "
                                       "This is the meter, not a broken key.") from None
            else:
                raise

print("Client ready.")

## 1. Your first programmatic call

Same model as the chat window — minus the app around it. Note the three things we control: the **model**, the **contents**, the **config**.

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents="In one sentence: why might a researcher prefer an API over a chat window?",
)
print(r.text)

## 2. Temperature — the randomness dial, demonstrated

Same prompt, five runs, twice: once at temperature 1 (default-ish), once at 0. Watch what happens to the variation. **This is the answer to Assignment 1, Exercise 2** — the instability you measured has a dial.

(The cell waits a few seconds between calls — ten polite calls on a free key take about 90 seconds.)

In [ ]:
prompt = ("Classify this app review's sentiment as exactly one word, "
          "positive/negative/mixed: 'Great features when it works, which is rarely.'")
for temp in [1.0, 0.0]:
    answers = []
    for _ in range(5):
        r = ask(prompt, temperature=temp)
        answers.append(r.text.strip())
        time.sleep(PACE)
    print(f"temperature {temp}: {answers}")

## 3. Structured output — answers as data, not prose

We ask for JSON and force it with `response_mime_type`. The result drops straight into a table. This is the pattern behind every pipeline in this course.

In [ ]:
import json, pandas as pd
abstract = open("../a1/abstracts.md", encoding="utf-8").read().split("## Abstract 01")[1].split("## Abstract 02")[0]
r = ask(f"""Extract from the abstract below. Verbatim quote for key_finding.
Any field not stated -> the string NOT REPORTED.
Return JSON with keys: research_question, method, sample, key_finding.
ABSTRACT: {abstract}""",
        temperature=0, response_mime_type="application/json")
row = json.loads(r.text)
pd.DataFrame([row])

## 4. Batching — many items per call

Rate limits meter *requests per minute* (and tokens), so we send 10 texts in one request and get 10 labeled rows back. On a free-tier key this pattern is what keeps you comfortable; on Vertex credit it's still just faster.

In [ ]:
sample = pd.read_csv("../data/sample_200.csv").head(10)
numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(sample.concern_text))
codebook = open("../data/codebook_concerns.md", encoding="utf-8").read()
r = ask(f"""Using ONLY the codebook below, assign one category (JOB/SURV/SKILL/FAIR/REL) to each numbered text.
Return a JSON list of objects with keys: n, category.
CODEBOOK:\n{codebook}\n\nTEXTS:\n{numbered}""",
        temperature=0, response_mime_type="application/json")
labels = pd.DataFrame(json.loads(r.text))
out = sample.reset_index(drop=True).assign(n=range(1, len(sample) + 1)).merge(labels, on="n", how="left")
out[["resp_id", "concern_text", "category"]]

## 5. What a call costs — the mental model

APIs charge per **token** (~3/4 of a word). Count before you run, and numbers never surprise you. (Counting tokens is free — it doesn't use your generation quota.)

In [ ]:
toks = client.models.count_tokens(model=MODEL, contents=numbered)
print(f"Those 10 texts = {toks.total_tokens} input tokens.")
print(f"All 1,200 survey rows, extrapolated: ~{toks.total_tokens * 120:,} tokens -> free on Flash's free tier; cents on paid.")

**Take-home experiment (10 min, optional):** change the batch in section 4 to rows 11–20 and rerun. Then look at one label you disagree with — you've just previewed Assignment 3's verification step V3.